# 🍕 Ejercicios de limpieza de datos: la pizzería

Esta vez trabajamos con los **pedidos de una pizzería**. Los datos tienen errores a propósito 😈:

- Faltan datos (huecos vacíos) en varias columnas.
- Hay un pedido repetido.
- Los nombres de las pizzas están escritos de formas distintas.
- El **precio** está guardado como texto, con el símbolo `€` y con **coma** decimal (`"9,50 €"`).
- Hay valores imposibles: unidades negativas o iguales a 0, y una valoración de 9 cuando la escala va de 1 a 5.

**¿Cómo funciona?**

1. Ejecuta la celda de los datos (▶️).
2. En cada ejercicio, escribe el código tú mismo en la celda vacía.
3. Ejecuta la celda y comprueba si el resultado tiene sentido.

Esta vez **no hay pistas** 💪. Si te atascas, repasa el Colab de limpieza de datos o el cuaderno anterior.

## 📦 Paso 0: Cargar los datos
Ejecuta esta celda sin cambiar nada.

In [ ]:
import pandas as pd
import numpy as np

datos = {
    "cliente":   ["Ana", "Raúl", "Carmen", "Iván", "Nuria", "Raúl", "Óscar", "Lola", "Hugo", "Irene", "Marcos", "Sara"],
    "pizza":     ["Margarita", " margarita", "BARBACOA", "Cuatro Quesos", "barbacoa ", " margarita", "cuatro quesos", "Margarita", "Barbacoa", "CUATRO QUESOS", "margarita", "Barbacoa"],
    "unidades":  [2, 1, 3, None, 1, 1, 2, -2, 1, 4, 0, 2],
    "precio":    ["9,50 €", "9,50 €", "11,00 €", "12,50 €", None, "9,50 €", "12,50 €", "9,50 €", "11,00 €", "12,50 €", "9,50 €", "11,00 €"],
    "valoracion":[5, 4, 3, 5, 4, 4, 9, 2, None, 5, 3, 4],
    "barrio":    ["Centro", "Norte", "Centro", None, "Sur", "Norte", "Centro", "Sur", "Norte", "Centro", "Sur", None],
}

df = pd.DataFrame(datos)
df

## Ejercicio 1: Mirar los datos 👀
Muestra las **5 primeras filas** de la tabla y después la **información general** (filas, columnas, tipos y valores no vacíos).

In [ ]:
display(df.head(10))
display(df.info())


❓ **Pregunta:** ¿De qué tipo es la columna `precio`? ¿Y `unidades`? ¿Por qué `unidades` no es un número entero (`int`)?

## Ejercicio 2: Contar los datos que faltan 🕳️
Cuenta cuántos valores vacíos hay en cada columna y, después, **cuántos hay en total** en toda la tabla.

In [ ]:
display(df.isna().sum())
print("Total del na en DF es: ", df.isna().sum().sum())

❓ **Pregunta:** ¿Qué columna tiene más huecos?

## Ejercicio 3: Quitar pedidos repetidos 👯
Primero **cuenta cuántas filas duplicadas** hay. Después elimínalas y muestra cuántas filas tiene la tabla ahora.

In [ ]:
df_original = df.copy()
display(df[df.duplicated(keep=False)].sort_index())
print("Numero de filas duplicadas: ", df.duplicated().sum())
df = df.drop_duplicates(ignore_index=True)
print("Numero de filas duplicadas: ", df.duplicated().sum())

❓ **Pregunta:** ¿Qué cliente estaba repetido?

## Ejercicio 4: Arreglar los nombres de las pizzas ✏️
Deja los nombres sin espacios sobrantes y con el formato `"Cuatro Quesos"` (primera letra de cada palabra en mayúscula). Después comprueba cuántas veces aparece cada pizza.

In [ ]:
print("Previo al refactor: ", df["pizza"].unique().tolist())
df["pizza"] = (df["pizza"].str.strip().str.title())
print("Despues del refactor: ",df["pizza"].unique().tolist())
print("Cantidad por tipo de pizza", df["pizza"].value_counts())

❓ **Pregunta:** ¿Cuántas pizzas distintas quedan?

## Ejercicio 5: Convertir el precio en número 💶
El precio es texto: `"9,50 €"`. Para poder hacer cuentas tienes que:

1. Quitar el símbolo `€` y los espacios.
2. Cambiar la coma por un punto.
3. Convertir la columna a número decimal (`float`).

Al final, muestra el tipo de la columna para comprobar que ha cambiado.

In [ ]:
display("Previo al refactor",df["precio"].head(5))
df["precio"] = (
    df["precio"]
        .str.replace("€","")
        .str.replace(",", ".")
        .astype(float)
    )
display("Despues del refactor",df["precio"].head(5))

❓ **Pregunta:** ¿Qué ha pasado con el precio que estaba vacío? ¿Ha dado error?

## Ejercicio 6: Quitar valores imposibles 🚫
Quédate solo con las filas que tienen sentido:

- Las **unidades** tienen que ser mayores que 0.
- La **valoración** tiene que estar entre 1 y 5 (ambos incluidos).

Ojo: **conserva las filas con huecos** en `unidades` o en `valoracion`, porque los rellenaremos después.

In [ ]:
filUndImpo = (df["unidades"] < 0)
filValImpo = ((df["valoracion"] < 1) | (df["valoracion"] > 5))
dfUndImposibles = df[filUndImpo]
dfValImposibles = df[filValImpo]

display("Las unidades imposibles son: ", dfUndImposibles)
display("Las valoraciones imposibles son: ", dfValImposibles)

df.loc[filUndImpo, "unidades"] = np.nan
df.loc[filValImpo, "valoracion"] = np.nan

display(df[["cliente","unidades", "valoracion"]])

❓ **Pregunta:** ¿Qué clientes han desaparecido? ¿Por qué ha desaparecido cada uno?

## Ejercicio 7: Rellenar los huecos 🧩
Rellena los datos que faltan así:

- `unidades` → con la **mediana** de las unidades.
- `precio` → con la **media** de los precios, **redondeada a 2 decimales**.
- `valoracion` → con la **mediana** de las valoraciones.
- `barrio` → con el **barrio que más se repite** (la moda). Es una columna de texto, así que no podemos usar media ni mediana.

In [ ]:
mediana_unidades = df["unidades"].median()
media_precios = df["precio"].mean()
mediana_valoracion = df["valoracion"].median()
moda_barrio = df["barrio"].mode()[0]
df_fillna = df.copy()
df_isna = df.copy()

In [ ]:
display(df)
df_isna.loc[df["unidades"].isna(), "unidades"] = mediana_unidades
df_isna.loc[df["precio"].isna(), "precio"] = media_precios
df_isna.loc[df["valoracion"].isna(), "valoracion"] = mediana_valoracion
df_isna.loc[df["barrio"].isna(), "barrio"] = moda_barrio

display(df_isna)

df_fillna["unidades"] = df_fillna["unidades"].fillna(mediana_unidades)
df_fillna["precio"] = df_fillna["precio"].fillna(media_precios)
df_fillna["valoracion"] = df_fillna["valoracion"].fillna(mediana_valoracion)
df_fillna["barrio"] = df_fillna["barrio"].fillna(moda_barrio)
display(df_fillna)

display("Previo al cambio", df)
df["unidades"] = df["unidades"].fillna(mediana_unidades)
df["precio"] = df["precio"].fillna(media_precios)
df["valoracion"] = df["valoracion"].fillna(mediana_valoracion)
df["barrio"] = df["barrio"].fillna(moda_barrio)
display("Despues del cambio",df)

## Ejercicio 8: Columnas nuevas y tipos finales 🛠️
1. Convierte `unidades` y `valoracion` a número **entero** (`int`). Ahora sí se puede, porque ya no hay huecos.
2. Crea una columna nueva llamada `total` con el importe de cada pedido (unidades × precio).

In [ ]:
display("previo al cambio de tipo: ", df.head())
display(df[["unidades", "valoracion"]].info())
df["unidades"] = (df["unidades"].astype(int))
df["valoracion"] = (df["valoracion"].astype(int))
df["total"] = (df["unidades"] * df["precio"])
display("despues del cambio de tipo: ", df.head())
display(df[["unidades", "valoracion"]].info())

❓ **Pregunta:** ¿Por qué no podíamos convertir `unidades` a `int` en el ejercicio 1?

## Ejercicio 9: Comprobación final ✅
Comprueba que ya no queda ningún hueco y muestra los tipos de todas las columnas.

In [ ]:
display(df)
display(df.info())

🎉 Si no queda ningún hueco, `precio` es `float` y `unidades` es `int`, ¡has limpiado la tabla!

## 🏆 Reto extra (opcional)
Con la tabla ya limpia, responde usando código:

1. ¿Cuánto dinero ha facturado la pizzería en total?
2. ¿Qué pizza ha vendido más **unidades**?
3. ¿Cuál es la valoración media de cada pizza? Ordénalas de mejor a peor.
4. ¿Cuánto se ha facturado en cada barrio?
5. Muestra solo los pedidos de más de 20 €.

In [ ]:
print("Facturacion total: ", df["total"].sum())

In [ ]:
display("Pizza con mayor unidades vendidas", df.groupby("pizza")["unidades"].sum().sort_values(ascending=False))

In [ ]:
display("Valoracion media de las pizzas", df.groupby("pizza")["valoracion"].sum().sort_values(ascending=False))

In [ ]:
display("Facturacion por barrio: ", df.groupby("barrio")["total"].sum().sort_values(ascending=False))

In [ ]:
display("Filtro de pedidos de mas de 20 euros: ", df[(df["total"] >20)])